# Production Debugging

Time-travel to exact moments of user complaints and examine agent memory state for debugging.

**Problem**: User reports "agent acted weird yesterday" but impossible to debug without seeing the exact memory state that influenced the decision.

In [ ]:
import asyncio
from datetime import datetime, timedelta
from memoir.core.memory import ProllyTreeMemoryStoreManager
from memoir.classifier.intelligent import IntelligentClassifier
from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(model="gpt-4", temperature=0)
classifier = IntelligentClassifier(llm=llm)
memory = ProllyTreeMemoryStoreManager(classifier=classifier)

In [ ]:
# Simulate production timeline
day1_commit = await memory.store_memory(
    "User prefers minimal UI designs",
    metadata={"message": "Day 1: UI preference", "timestamp": "2024-01-15T09:00:00Z"}
)

day2_commit = await memory.store_memory(
    "User likes dark mode for coding",
    metadata={"message": "Day 2: Theme preference", "timestamp": "2024-01-16T14:30:00Z"}
)

print(f"Production timeline established")
print(f"Day 1 commit: {day1_commit[:8]}")
print(f"Day 2 commit: {day2_commit[:8]}")

In [ ]:
# Day 3: Problem occurs - agent gives bad recommendation
problem_commit = await memory.store_memory(
    "Agent recommended bright neon colors for dashboard",  # This conflicts with minimal UI preference!
    metadata={"message": "Day 3: Bad recommendation", "timestamp": "2024-01-17T11:15:00Z"}
)

print(f"Problem occurred at commit: {problem_commit[:8]}")

In [ ]:
# Day 4: User reports issue
user_report_time = "2024-01-18T10:00:00Z"
print(f"User reports: 'Agent recommended terrible colors yesterday at 11:15 AM'")
print(f"Report timestamp: {user_report_time}")

In [ ]:
# Production debugging: Time-travel to exact moment
print("\nDebugging: Memory state at moment of bad recommendation")
problem_memories = await memory.search_memories(
    "UI design preferences", 
    at_commit=problem_commit
)

print("Agent's memory when making recommendation:")
for mem in problem_memories[:3]:
    print(f"- {mem.content}")

In [ ]:
# Examine taxonomy state at problem point
print("\nAnalyzing memory structure at problem time...")
taxonomy_paths = await memory.get_taxonomy_paths()
ui_paths = [path for path in taxonomy_paths if 'ui' in path.lower() or 'design' in path.lower()]
print(f"UI-related paths in taxonomy: {len(ui_paths)}")
for path in ui_paths[:3]:
    print(f"  - {path}")

In [ ]:
# Root cause analysis: Check what agent "saw" before bad decision
pre_problem_memories = await memory.search_memories(
    "UI design preferences", 
    at_commit=day2_commit
)

print("\nMemory state BEFORE problem (should include minimal UI preference):")
for mem in pre_problem_memories[:3]:
    print(f"- {mem.content}")

print(f"\nRoot cause: Agent had correct preferences but bad logic override")

In [ ]:
# Create debug branch from problem point
await memory.checkout(day2_commit)  # Go to last good state
await memory.create_branch("debug_fix")
await memory.checkout("debug_fix")

print(f"Created debug branch from clean state: {day2_commit[:8]}")

In [ ]:
# Test fix in debug branch
fix_commit = await memory.store_memory(
    "Agent recommended subtle grays and whites for minimal dashboard design",
    metadata={"message": "Corrected recommendation"}
)

print(f"Testing fix in debug branch: {fix_commit[:8]}")

# Verify fix
fix_verification = await memory.search_memories("dashboard design recommendation")
print("Fixed recommendation:")
print(f"- {fix_verification[0].content}")

In [ ]:
# Deploy fix: Merge back to main
await memory.checkout("main")
await memory.merge("debug_fix")

print("Fix deployed to production")

# Verify production state
final_check = await memory.search_memories("UI design preferences")
print("\nProduction memory after fix:")
for mem in final_check[:3]:
    print(f"- {mem.content}")

In [ ]:
# Cleanup debug branch
await memory.delete_branch("debug_fix")
print("\nDebug branch cleaned up")
print("Production debugging complete: Issue identified, fixed, and deployed")